# Résoudre `4_precise` avec OR-Tools
Quatrième jeu de données : 8 000 modèles et 40 000 datasets. Les intervalles de besoins sont **très étroits** (0 à 10 unités) : il faut composer des tailles **presque exactes**.

**Version 3.** Ce notebook n'implémente plus rien lui-même : il appelle le package `ia_responsable` (dossier `src/`). Le récit et les expériences sont ceux de la version précédente ; le code est celui que teste `make test` et qu'utilise `make solve`. Les fonctions sont commentées dans les modules indiqués entre parenthèses.

**Durée d'exécution** : environ 20 minutes (15 pour le solveur). Pour un essai rapide, mettez `SOLVER_SECONDS = 300` en partie 4.

## 1. Charger l'instance

In [1]:
from pathlib import Path
import json, time, random
from collections import Counter
import pandas as pd

from ia_responsable import load
from ia_responsable.cli import official_score
from ia_responsable.fill import fill, give, new_pools, no_full_models
from ia_responsable.model import solve
from ia_responsable.packing import REFERENCE_ORDERS, noisy_orders, pack
from ia_responsable.pipeline import plan_value, run
from ia_responsable.repair import State, complete, repair

pd.set_option("display.width", 150)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
inst = load(ROOT / "datasets" / "4_precise.json")
WINNER = 1_031_941


def score(plan):
    """Score officiel (test_solution.py). Lève une erreur si le plan est invalide."""
    s, valid, message = official_score(ROOT, inst.text, plan)
    assert valid, message
    return s


def solver_row(info, sel):
    return dict(statut=info.status, objectif=info.objective, borne=round(info.bound, 1), écart_prouvé=round(info.gap, 1),
                secondes=round(info.seconds), pleins=len(sel.full), demi=len(sel.half), sources=len(sel.sources))


def failed_full(sel, plan):
    """Modèles choisis à pleine valeur qui n'ont pas eu leurs données libres."""
    data = {}
    for d, m in plan["dataMappings"]:
        data.setdefault(m, []).append(d)
    return [m for m in sel.full if m not in data or any(inst.copyrighted[d] for d in data[m])]


def save_if_better(plan):
    out = ROOT / "solutions" / f"{inst.name}_main.json"
    new = score(plan)
    current = official_score(ROOT, inst.text, json.loads(out.read_text()))[0] if out.exists() else 0
    if new > current:
        out.write_text(json.dumps(plan))
        print(f"{out.name} mis à jour : {current:,} -> {new:,}")
    else:
        print(f"{out.name} inchangé ({current:,}) ; ce notebook a obtenu {new:,}")


print(f"{inst.name} : {len(inst.value)} modèles, {len(inst.size)} datasets | gagnant du concours : {WINNER:,}")

4_precise : 8000 modèles, 40000 datasets | gagnant du concours : 1,031,941


## 2. Ce qui change

In [2]:
print("besoins par modèle :", dict(sorted(Counter(len(n) for n in inst.needs.values()).items())))
print(f"sources possibles : {len(inst.mono)} | modèles impossibles : {len(inst.impossible)}")
print(f"datasets sous droits : {sum(inst.copyrighted.values()) / len(inst.size):.0%}")
widths = Counter(hi - lo for needs in inst.needs.values() for t, lo, hi in needs if lo <= hi)
print("largeur des intervalles :", dict(sorted(widths.items())))

besoins par modèle : {2: 3206, 3: 3196, 4: 1598}
sources possibles : 0 | modèles impossibles : 380
datasets sous droits : 50%
largeur des intervalles : {0: 1102, 1: 2121, 2: 2153, 3: 2137, 4: 2194, 5: 2177, 6: 2244, 7: 2152, 8: 2135, 9: 2197, 10: 1106}


**Lecture**
- **Aucun modèle n'a un seul besoin** : aucun ne peut servir de source. Les 380 modèles impossibles sont **définitivement exclus**.
- La moitié des datasets est sous droits : la rareté des données libres revient.
- Plus de 1 000 besoins exigent une taille **exacte**.

## 3. Composer une taille précise (`fill.py`)
Pour finir un remplissage dans une fenêtre de quelques unités, un seul dataset de la bonne taille est rare, mais **deux** datasets dont la somme tombe juste sont nombreux : avec 5 000 datasets d'un type, il y a environ 12 millions de paires. `fill` cherche une paire par la méthode des **deux pointeurs**, en un seul parcours de la liste triée.

In [3]:
m, (t, lo, hi) = next((m, n) for m, needs in inst.needs.items() for n in needs if n[1] == n[2])
pools = new_pools(inst)
got = fill(pools[(t, 0)], lo, hi)
print(f"modèle {m}, besoin {t} = exactement {lo} -> tailles {[inst.size[d] for d in got]}, somme {sum(inst.size[d] for d in got)}")

modèle 4, besoin n = exactement 11641 -> tailles [10000, 1641], somme 11641


Chaque besoin, **pris isolément**, peut-il être rempli ? On les essaie tous, en remettant les datasets après chaque essai.

In [4]:
t0 = time.time()
pools = {t: new_pools(inst)[(t, 0)] for t in "ntic"}
for t in "ntic":                        # une réserve par type, libres et sous droits réunis
    for d in new_pools(inst)[(t, 1)].ids:
        pools[t].put(d)
n_needs, failures, n_datasets = 0, [], Counter()
for m, needs in inst.needs.items():
    for t, lo, hi in needs:
        if lo <= hi:
            n_needs += 1
            got = fill(pools[t], lo, hi)
            if got is None:
                failures.append((m, t))
            else:
                n_datasets[len(got)] += 1
                for d in got:
                    pools[t].put(d)
print(f"{n_needs} besoins testés en {time.time() - t0:.0f} s | irréalisables : {len(failures)}")
print("datasets par besoin :", dict(sorted(n_datasets.items())))

21718 besoins testés en 24 s | irréalisables : 0
datasets par besoin : {1: 5622, 2: 7177, 3: 5420, 4: 2274, 5: 1090, 6: 135}


**Lecture** : **tous** les besoins sont réalisables isolément. La précision n'est pas un obstacle en soi : la difficulté viendra de la **concurrence** entre modèles.

## 4. Le modèle, et deux corrections nées de ce dataset
1. **Un bug.** Dans une première version, la contrainte « une source par besoin impossible » n'était écrite que si au moins une source candidate existait. Ici, il n'y en a aucune : la contrainte n'était jamais écrite, et le solveur entraînait gratuitement 173 modèles impossibles (objectif 1 035 477, bien au-dessus du gagnant). **Un score trop beau cache souvent une contrainte oubliée.** Le package écrit la contrainte même quand la somme est vide (`model.py`).
2. **La mise à l'échelle de l'énergie.** Le package exprime l'énergie en milliards : coefficients de l'ordre de $10^3$ au lieu de $10^{12}$ (voir le notebook 14). Ici, le gain est faible (une dizaine de points à durée égale) : la difficulté vient surtout de la taille de la recherche.

In [5]:
SOLVER_SECONDS = 900
sel, info = solve(inst, seconds=SOLVER_SECONDS)
pd.DataFrame([solver_row(info, sel)], index=[f"SCIP, {SOLVER_SECONDS} s"])

,statut,objectif,borne,écart_prouvé,secondes,pleins,demi,sources
"SCIP, 900 s",FEASIBLE,1031919,1031946.4,27.4,901,2177,191,0


**Lecture** : la borne est d'environ 1 031 947, à 6 points du gagnant. En 15 minutes, SCIP s'en approche sans prouver l'optimum. **Avec une limite de temps, le résultat n'est pas reproductible d'une machine à l'autre** : le solveur explore plus ou moins de nœuds selon la vitesse du processeur.

## 5. Plusieurs ordres d'attribution
Les deux ordres de référence, plus **40 ordres bruités** (« petits besoins » avec un facteur aléatoire de ±20 % par modèle, `packing.noisy_orders`). C'est une **recherche à départs multiples** : un glouton déterministe donne toujours la même réponse ; un peu de hasard permet d'en explorer plusieurs. Chaque attribution est complétée et réparée, en essayant **tous** les partenaires d'échange.

In [6]:
orders = dict(REFERENCE_ORDERS) | noisy_orders(inst, 40, seed=0)
rows, best = [], (0, None, None)
t0 = time.time()
for name, key in orders.items():
    plan, n_failed = pack(inst, sel, key)
    state = State(inst, plan)
    complete(state)
    n = repair(state, max_partners=None)
    s = score(state.plan())
    rows.append(dict(ordre=name, ratés=n_failed, avant_réparation=score(plan), réparations=n, score=s, écart_au_gagnant=s - WINNER))
    if s > best[0]:
        best = (s, name, state.plan())
print(f"{len(orders)} ordres en {time.time() - t0:.0f} s")
table = pd.DataFrame(rows).set_index("ordre")
table.sort_values("score", ascending=False).head(10)

42 ordres en 184 s


,ratés,avant_réparation,réparations,score,écart_au_gagnant
ordre,,,,,
bruité 18,1,1031425,2,1031901,-40
bruité 28,1,1031445,3,1031894,-47
bruité 11,2,1030951,3,1031893,-48
plus contraints,1,1031420,2,1031892,-49
bruité 6,2,1030937,3,1031892,-49
bruité 26,1,1031430,2,1031888,-53
petits besoins,2,1030928,2,1031886,-55
bruité 38,1,1031430,3,1031884,-57
bruité 22,1,1031425,2,1031882,-59


In [7]:
print(f"de {table.score.min():,} à {table.score.max():,} selon l'ordre")
best_score, best_name, best_plan = best
print(f"meilleur ordre : {best_name} | {best_score:,} | écart au gagnant : {best_score - WINNER:+,} ({best_score / WINNER:.4%})")
print(f"sélection du solveur : {info.objective:,} | borne : {info.bound:,.1f}")
save_if_better(best_plan)

de 1,031,834 à 1,031,901 selon l'ordre
meilleur ordre : bruité 18 | 1,031,901 | écart au gagnant : -40 (99.9961%)
sélection du solveur : 1,031,919 | borne : 1,031,946.4


4_precise_main.json inchangé (1,031,910) ; ce notebook a obtenu 1,031,901


**Lecture**
- Selon l'ordre, un ou deux modèles pleins ne trouvent pas leurs données libres, et le résultat final varie de plusieurs dizaines de points : **l'attribution est une loterie**, et plusieurs tirages valent mieux qu'un seul.
- L'écart restant se partage entre la sélection du solveur (pas encore optimale) et l'attribution.
- Pistes essayées sans succès : le recuit du notebook 10 en finition (aucun gain) et la boucle du notebook 11. Ici, la boucle empire les choses : à chaque itération, **d'autres** modèles échouent, et la borne baisse. Les échecs viennent de la concurrence entre tous, pas de quelques modèles particuliers.

## À retenir
1. **Regarder la structure avant de modéliser** : sans modèle à un seul besoin, les sources disparaissent.
2. **Un score trop beau cache une contrainte oubliée** : une boucle sur une liste vide n'écrit aucune contrainte.
3. **La précision n'est pas le vrai problème, la concurrence l'est.**
4. **Plusieurs départs valent mieux qu'un.**
5. **Avec une limite de temps, un solveur n'est pas reproductible** d'une machine à l'autre.